# Maia Corpus v8 — Integrity & Interruption Recovery Audit v1

**Purpose:** independently reconstruct the **existing pilot** from source documents; prove exact split isolation and token equality (including pending tails); run a **non-destructive crash/restart experiment** using the original preparation implementation in an isolated temporary directory.

**CPU only.** No production processing. The existing pilot, corpus, tokenizer, and frozen manifest are read-only. Temporary crash-test outputs are created under `/content/maia_recovery_audit_*`, **never** in the production directory. This notebook does not automatically set `PRODUCTION_MODE=True`.

**Caveats:** hash-disjointness is exact-text only (not semantic/near-duplicate). This tests interruption after a shard rename, not every conceivable filesystem or Drive failure. Source metadata and tokenizer are validated, but the original corpus freeze provenance remains independently unresolved.

In [ ]:
from pathlib import Path
import os, sys, json, gzip, hashlib, sqlite3, time, tempfile, shutil, unicodedata, collections, datetime
import numpy as np
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab; provide accessible paths below.')
try:
    from transformers import AutoTokenizer
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'tokenizers'])
    from transformers import AutoTokenizer

ROOT = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/corpus_v8/runs/acquisition_v8_5_001')
PILOT = ROOT / 'pretraining_preparation_v1'
TOKENIZER_DIR = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer')
# Upload the exact original notebook to Colab if not present at this path.
ORIGINAL_NOTEBOOK = Path('/content/Maia_Corpus_v8_Production_Preparation_v1.ipynb')
RUN_CRASH_TEST = True
MAX_REFERENCE_DOCS = 3000  # guard: pilot should be far below this
assert (PILOT/'run_config.json').is_file(), 'Pilot configuration missing'
assert (PILOT/'state.sqlite3').is_file(), 'Pilot SQLite state missing'
assert TOKENIZER_DIR.is_dir(), 'Tokenizer missing'
print('Pilot:', PILOT)
print('Original notebook for isolated crash test:', ORIGINAL_NOTEBOOK)

## A. Independent reconstruction of the pilot

We use the immutable `run_config.json` and `state.sqlite3`, independently read every selected source chunk, reproduce NFC normalization, exact-text hash split, EOS insertion, and compare **every token ID** against the concatenation of all committed shards and the pending tail. This also checks source/size/mtime, SQLite consistency, per-chunk counts, and exact-hash split isolation. The comparison is deliberately independent of the original notebook's `process_chunk` implementation.

In [ ]:
cfg = json.loads((PILOT/'run_config.json').read_text(encoding='utf-8'))
assert cfg['production_mode'] is False, 'This audit is designed for the completed PILOT'
assert cfg['block_size'] == 2048
assert cfg['split_percent'] == {'train':94,'validation':3,'test':3}
assert cfg['eos_per_document'] is True
files = cfg['tokenizer_file_sha256']
for name, expected in files.items():
    actual=hashlib.sha256((TOKENIZER_DIR/name).read_bytes()).hexdigest()
    assert actual == expected, f'Tokenizer mismatch: {name}'
tok = AutoTokenizer.from_pretrained(str(TOKENIZER_DIR),local_files_only=True,use_fast=True)
assert len(tok)==50257 and tok.eos_token_id==2
con=sqlite3.connect(f'file:{PILOT/"state.sqlite3"}?mode=ro',uri=True)
assert con.execute('PRAGMA quick_check').fetchone()[0]=='ok'
inv=cfg['selected_inventory']
assert len(inv)>0
assert len(inv)==con.execute('SELECT COUNT(*) FROM completed_chunk').fetchone()[0], 'Pilot has unfinished chunks'
print('Selected chunks:',len(inv),'Tokenizer SHA-256: PASS','SQLite: PASS')

In [ ]:
def canon(s): return unicodedata.normalize('NFC',s).strip()
def dh(s): return hashlib.sha256(s.encode('utf-8')).hexdigest()
def split_hash(h):
    bucket=int.from_bytes(hashlib.sha256((cfg['split_seed']+':'+h).encode()).digest()[:8],'big')%10000
    if bucket < cfg['split_percent']['train']*100: return 'train'
    if bucket < (cfg['split_percent']['train']+cfg['split_percent']['validation'])*100: return 'validation'
    return 'test'

expected=collections.defaultdict(list)
seen={}; source_counts={}; checked_docs=0; checked_dups=0
for item in inv:
    path=Path(item['path']); assert path.is_file(),f'Missing source {path}'
    st=path.stat()
    assert (st.st_size,st.st_mtime_ns)==(item['size'],item['mtime_ns']),f'Source metadata changed {path}'
    docs=dups=tokens=0
    with gzip.open(path,'rt',encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            obj=json.loads(line)
            raw=obj.get('text')
            assert isinstance(raw,str),f'Missing text in {path}'
            text=canon(raw)
            if not text: continue
            h=dh(text)
            if h in seen:
                dups+=1; continue
            split=split_hash(h)
            seen[h]=split
            ids=tok.encode(text,add_special_tokens=False)+[tok.eos_token_id]
            expected[(item['source'],split)].extend(ids)
            docs+=1; tokens+=len(ids)
            checked_docs+=1
            assert checked_docs <= MAX_REFERENCE_DOCS, 'Pilot larger than expected; raise MAX_REFERENCE_DOCS intentionally'
            # Original pilot stops after unique+duplicate count reaches its per-chunk cap.
            if docs+dups>=cfg['max_docs_per_chunk_pilot']: break
    actual=con.execute('SELECT docs,duplicates,tokens FROM completed_chunk WHERE path=?',(str(path),)).fetchone()
    assert actual==(docs,dups,tokens),(path,actual,(docs,dups,tokens))
    checked_dups+=dups
    source_counts[str(path)]=(docs,dups,tokens)
print('Independent documents:',checked_docs,'duplicates:',checked_dups)
assert len(seen)==con.execute('SELECT COUNT(*) FROM seen').fetchone()[0]
for h,split in con.execute('SELECT hash,split FROM seen'):
    assert seen[h]==split, f'Split/hash mismatch {h}'
print('Global exact-hash split isolation: PASS')

In [ ]:
# Independently decode committed shards + pending tails and compare exact token IDs.
from array import array
import struct
assert con.execute('SELECT COUNT(*) FROM (SELECT hash FROM seen GROUP BY hash HAVING COUNT(*)>1)').fetchone()[0]==0
summary=[]
for source in cfg['source_names']:
    for split in ('train','validation','test'):
        got=[]; last=-1
        for index,path,blocks,sha in con.execute('SELECT shard_index,path,blocks,sha256 FROM shard WHERE source=? AND split=? ORDER BY shard_index',(source,split)):
            assert index==last+1, f'Shard index gap: {source}/{split}'
            last=index
            p=Path(path)
            assert p.is_file() and p.stat().st_size==blocks*cfg['block_size']*4
            data=p.read_bytes()
            assert hashlib.sha256(data).hexdigest()==sha
            got.extend(np.frombuffer(data,dtype='<u4').tolist())
        row=con.execute('SELECT ids FROM pending WHERE source=? AND split=?',(source,split)).fetchone()
        tail=np.frombuffer(row[0],dtype='<u4').tolist() if row else []
        assert len(tail)<cfg['block_size']
        got.extend(tail)
        ref=expected[(source,split)]
        assert got==ref, f'TOKEN MISMATCH {source}/{split}: actual={len(got)} expected={len(ref)}'
        summary.append({'source':source,'split':split,'tokens':len(got),'tail':len(tail)})
assert sum(x['tokens'] for x in summary)==con.execute('SELECT COALESCE(SUM(tokens),0) FROM completed_chunk').fetchone()[0]
print('Exact token reconstruction (shards + tails): PASS')
print('Conservation, per-source and per-split: PASS')
print('Validated tokens:',sum(x['tokens'] for x in summary))
for x in summary: print(x)

## B. Isolated crash/restart test — actual original `process_chunk`

This test executes the **original code** for `process_chunk` and its helpers in a temporary directory. It constructs two small synthetic JSONL.GZ chunks, uses the validated real tokenizer, and runs two experiments:

1. Uninterrupted reference execution.
2. Injected crash **after an atomic shard rename but before the SQLite transaction**. The expected orphan is detected, moved to an isolated quarantine directory after verifying no committed row refers to it, and the original chunk is retried. Final token streams and SQLite records must match the reference run exactly.

**Why quarantine?** The original pipeline intentionally refuses to overwrite orphan shards. This test demonstrates a safe, explicit recovery procedure; it does not claim the original pipeline automatically repairs arbitrary crashes.

In [ ]:
import nbformat, types
assert ORIGINAL_NOTEBOOK.is_file(), ('Upload the original preparation notebook to Colab as '
                                    '/content/Maia_Corpus_v8_Production_Preparation_v1.ipynb')
original=nbformat.read(str(ORIGINAL_NOTEBOOK),as_version=4)
helper_src=''.join(original.cells[8].source)
process_src=''.join(original.cells[9].source)
assert 'def process_chunk(' in process_src and 'def shard_file(' in helper_src
assert 'os.replace(temp,target)' in process_src, 'Implementation changed; review fault injection'
print('Original process_chunk loaded:',len(process_src),'characters')

In [ ]:
# All files generated in this cell live in a fresh LOCAL temporary directory.
# We use two chunks of synthetic documents large enough to create full 2048-token shards.
work=Path(tempfile.mkdtemp(prefix='maia_recovery_audit_',dir='/content' if Path('/content').exists() else None))
source='fineweb2_pt'
chunks=[]
for chunk_idx in range(2):
    path=work/f'{source}_chunk_{chunk_idx:06d}.jsonl.gz'
    with gzip.open(path,'wt',encoding='utf-8') as f:
        for j in range(36):
            text=(f'Maia synthetic crash test chunk {chunk_idx} document {j}. '
                  + ('O treinamento usa documentos verificáveis e tokens reprodutíveis. '*14))
            f.write(json.dumps({'text':text},ensure_ascii=False)+'\n')
    chunks.append(path)

# Exact original functions, isolated namespace; no writes to PILOT/ROOT.
def new_env(out):
    out=Path(out)
    assert str(out.resolve()).startswith(str(work.resolve())+os.sep)
    out.mkdir(parents=True,exist_ok=True)
    db=sqlite3.connect(out/'state.sqlite3')
    db.execute('PRAGMA synchronous=FULL')
    db.executescript("""
CREATE TABLE seen(hash TEXT PRIMARY KEY, split TEXT NOT NULL);
CREATE TABLE completed_chunk(path TEXT PRIMARY KEY, source TEXT NOT NULL, docs INTEGER NOT NULL, duplicates INTEGER NOT NULL, tokens INTEGER NOT NULL, completed_utc TEXT NOT NULL);
CREATE TABLE shard(source TEXT NOT NULL, split TEXT NOT NULL, shard_index INTEGER NOT NULL, path TEXT NOT NULL UNIQUE, blocks INTEGER NOT NULL, sha256 TEXT NOT NULL, PRIMARY KEY(source,split,shard_index));
CREATE TABLE pending(source TEXT NOT NULL, split TEXT NOT NULL, ids BLOB NOT NULL, PRIMARY KEY(source,split));
"""); db.commit()
    inventory=[{'source':source,'path':str(p),'size':p.stat().st_size,'mtime_ns':p.stat().st_mtime_ns} for p in chunks]
    namespace=dict(Path=Path,OUT=out,con=db,np=np,gzip=gzip,json=json,os=os,hashlib=hashlib,
                   unicodedata=unicodedata,datetime=datetime.datetime,timezone=datetime.timezone,
                   SPLIT_SEED=cfg['split_seed'],SPLIT_PERCENT=cfg['split_percent'],BLOCK_SIZE=2048,
                   SHARD_MAX_BLOCKS=512,EOS_PER_DOCUMENT=True,PRODUCTION_MODE=False,
                   MAX_DOCS_PER_CHUNK_PILOT=100,tok=tok,config={'selected_inventory':inventory})
    exec(helper_src,namespace)
    exec(process_src,namespace)
    return namespace,db

reference,refdb=new_env(work/'reference')
for p in chunks: reference['process_chunk'](source,p)
reference['verify_committed_shards']()
print('Uninterrupted reference: PASS')

In [ ]:
# Inject a failure after the first successful rename to simulate interruption.
# The patched os is a local proxy, NOT the real os module, and can only touch work/.
class FaultOS:
    path=os.path
    fsync=staticmethod(os.fsync)
    def __init__(self): self.injected=False
    def replace(self,src,dst):
        assert str(Path(dst).resolve()).startswith(str(work.resolve())+os.sep)
        os.replace(src,dst)
        if not self.injected:
            self.injected=True
            raise RuntimeError('INJECTED_CRASH_AFTER_RENAME_BEFORE_SQLITE_COMMIT')

crashed,crashdb=new_env(work/'crashed')
fault=FaultOS()
crashed['os']=fault
try:
    crashed['process_chunk'](source,chunks[0])
except RuntimeError as exc:
    assert str(exc)=='INJECTED_CRASH_AFTER_RENAME_BEFORE_SQLITE_COMMIT'
else:
    raise AssertionError('Expected crash did not occur')
assert fault.injected
assert crashdb.execute('SELECT COUNT(*) FROM completed_chunk').fetchone()[0]==0
assert crashdb.execute('SELECT COUNT(*) FROM shard').fetchone()[0]==0
orphans=list((work/'crashed'/'shards').rglob('*.bin'))
assert orphans, 'Expected uncommitted shard after crash'
print('Crash injection: PASS; uncommitted shards:',len(orphans))
# Close the old connection to simulate a fresh process.
crashdb.close()

In [ ]:
# Inspect and quarantine ONLY uncommitted files inside isolated crash-test output.
crashdir=work/'crashed'
check=sqlite3.connect(crashdir/'state.sqlite3')
assert check.execute('PRAGMA quick_check').fetchone()[0]=='ok'
committed={Path(r[0]).resolve() for r in check.execute('SELECT path FROM shard')}
quarantine=work/'quarantine';quarantine.mkdir()
for p in list((crashdir/'shards').rglob('*')):
    if not p.is_file(): continue
    assert p.resolve() not in committed, 'Refusing to move a committed file'
    assert str(p.resolve()).startswith(str(crashdir.resolve())+os.sep)
    dest=quarantine/p.relative_to(crashdir)
    dest.parent.mkdir(parents=True,exist_ok=True)
    shutil.move(str(p),str(dest))
check.close()
# Reopen SQLite and retry from the original code; same synthetic input, same tokenizer.
# Do not call new_env here: that would recreate tables and replace the original connection.
recoverydb=sqlite3.connect(crashdir/'state.sqlite3')
recoverydb.execute('PRAGMA synchronous=FULL')
recovered=dict(crashed)
recovered['con']=recoverydb
recovered['os']=os
exec(helper_src,recovered)
exec(process_src,recovered)
for p in chunks: recovered['process_chunk'](source,p)
recovered['verify_committed_shards']()
assert recoverydb.execute('PRAGMA quick_check').fetchone()[0]=='ok'
print('Quarantine + resumed original process_chunk: PASS')

In [ ]:
def snapshot(ns,db):
    streams={}
    for split in ('train','validation','test'):
        ids=[]
        for (path,) in db.execute('SELECT path FROM shard WHERE source=? AND split=? ORDER BY shard_index',(source,split)):
            ids.extend(np.fromfile(path,dtype='<u4').tolist())
        row=db.execute('SELECT ids FROM pending WHERE source=? AND split=?',(source,split)).fetchone()
        if row: ids.extend(np.frombuffer(row[0],dtype='<u4').tolist())
        streams[split]=ids
    rows=list(db.execute('SELECT source,docs,duplicates,tokens FROM completed_chunk ORDER BY path'))
    hashes=list(db.execute('SELECT hash,split FROM seen ORDER BY hash'))
    return streams,rows,hashes

ref_snap=snapshot(reference,refdb)
rec_snap=snapshot(recovered,recoverydb)
assert ref_snap==rec_snap, 'Recovery does not reproduce reference streams/metadata'
assert sum(map(len,ref_snap[0].values()))==sum(row[3] for row in ref_snap[1])
print('Uninterrupted vs resumed token IDs: EXACT MATCH')
print('Uninterrupted vs resumed document hashes/splits: EXACT MATCH')
print('Uninterrupted vs resumed chunk counters: EXACT MATCH')
print('SQLite consistency: PASS')

## C. Audit report and production gate

The report is saved **locally** first. No original files are changed. The gate covers the properties tested here, not storage capacity, processing speed, licensing, tokenizer historical provenance, near-duplicate leakage, or actual production throughput. The next step is a *separate* limited production canary before full processing.

In [ ]:
report={
 'audit':'Maia Corpus v8 preparation integrity and crash-restart v1',
 'timestamp_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'pilot_config':str(PILOT/'run_config.json'),
 'pilot_documents':checked_docs,
 'pilot_duplicates':checked_dups,
 'pilot_tokens':sum(x['tokens'] for x in summary),
 'checks':{
  'sqlite_quick_check':'PASS',
  'tokenizer_hashes':'PASS',
  'source_metadata':'PASS',
  'independent_exact_token_reconstruction':'PASS',
  'pending_tails':'PASS',
  'global_exact_hash_split_isolation':'PASS',
  'chunk_document_and_token_accounting':'PASS',
  'injected_post_rename_crash':'PASS',
  'isolated_orphan_quarantine':'PASS',
  'resume_equivalent_to_uninterrupted':'PASS'
 },
 'production_gate':'PILOT_RECOVERY_VALIDATED_PRODUCTION_CANARY_REQUIRED',
 'limitations':['No near-duplicate check','No license audit','No full corpus freeze SHA recheck',
                'No end-to-end Drive crash simulation','No sustained production I/O benchmark']
}
report_path=work/'audit_report.json'
report_path.write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps(report,indent=2,ensure_ascii=False))
print('LOCAL REPORT:',report_path)
print('All production input/output files remained untouched.')

### What to send back

Send the JSON report printed by the last cell. If any assertion fails, **do not enable production mode**; send the complete traceback. If all pass, we can perform a bounded production canary and review disk quota, I/O rate, and orphan recovery before the full 21,021-chunk run.

**Note:** Upload the original `Maia_Corpus_v8_Production_Preparation_v1.ipynb` to the Colab runtime (`/content/`) before section B; the original notebook is used as the code under test.